# 🕵️ Homework 2: Exploratory Data Analysis (Continued), SQL

## ‼️ Due Date: Friday, September 18, 11:59 PM
You must submit this assignment to Pensive by the on-time deadline, Friday, September 18, 11:59 PM. Please read the syllabus for the Slip Day policy. No late submissions beyond the details in the Slip Day policy will be accepted. While course staff is happy to help you if you encounter difficulties with submission, we may not be able to respond to late-night requests for assistance (TAs need to sleep, after all!). **We strongly encourage you to plan to submit your work to Pensive several hours before the stated deadline.** This way, you will have ample time to contact staff for submission support. 

Please read the instructions carefully when you are submitting your work to Pensive.

## 💪 Collaboration Policy

Data science is a collaborative activity. While you may talk with others about the homework, we ask that you **write your solutions individually**. If you do discuss the assignments with others, please **include their names** below.

**Collaborators**: *list collaborators here*

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw02.ipynb")


## 📜 This Assignment

After this homework, you should be comfortable with:
* Reading `Polars` documentation and using `Polars` methods,
* Working with data at different levels of granularity,
* Using `group_by` with different aggregation functions,
* Chaining different `Polars` functions and methods to find answers to exploratory questions, and
* Using `SQL` to query multiple tables in a database


## 💯 Score Breakdown 
Question | Manual | Points
--- | --- | ---
1a | No | 2
1b | No | 3
2a | No | 2
2b | No | 3
3a | Yes | 1
3b | Yes | 1
3c | Yes | 1
3d | Yes | 1
3e | Yes | 1
3f | Yes | 1
4a | No | 1
4b | No | 1
4c | No | 1
4d | Yes | 1
4e | Yes | 2
4f | No | 1 
5a | No | 2 
5b | No | 2 
6 | No | 3 
7 | No | 3 
8 | No | 0
Total | 8 | 33

## ✊ Before You Start

For each question in the assignment, please write down your answer in the answer cell(s) right below the question. 

We understand that it is helpful to have extra cells breaking down the process towards reaching your final answer. If you happen to create new cells below your answer to run code, **NEVER** add cells between a question cell and the answer cell below it. It will cause errors when we run the autograder, and it will sometimes cause a failure to generate the PDF file.

Finally, unless we state otherwise, **do not use for loops or list comprehensions**. The majority of the Python questions can be done using built-in commands in `Polars` and `NumPy`. Our autograder isn't smart enough to check, but you're depriving yourself of key learning objectives if you write loops / comprehensions, and you also won't be ready for the midterm.

### 🐛 Debugging Guide
If you run into any technical issues, we highly recommend checking out the [Data 100 Debugging Guide](https://ds100.org/debugging-guide/). In this guide, you can find general questions about Jupyter notebooks / Datahub, Pensive, and common `Polars` errors.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import sqlalchemy
from pathlib import Path
import sql

plt.style.use('fivethirtyeight') # Use plt.style.available to see more styles
sns.set()
sns.set_context("talk")
np.set_printoptions(threshold=5) # Avoid printing out big matrices
%matplotlib inline
%load_ext sql

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 🐻‍❄️ Part 1: Continuing to Explore Food Safety Data

*For most of Part 1, we recommend minimal to no LLM usage, so that you get practice with the basics of `Polars`. One subpart will be marked as LLM-OK, and you are encouraged to use LLMs and/or documentation searches to help you with this question, as long as you follow the [course LLM policy](https://ds100.org/fa26/syllabus/#collaboration-policy-and-academic-honesty).*

In HW 1, we took you through the entire process of reading data from a file to perform some exploration of the data. Here, we again load the dataset that we will be using in HW 2 along with some of the columns we had added in HW 1. For any additional context regarding the dataset, we encourage you to revisit HW 1.

In [ ]:
bus = pl.read_csv('data/bus.csv', encoding='latin-1').rename({"business id column": "bid"})
bus = bus.with_columns(pl.col('postal_code').str.slice(0, 5).alias('postal5'))
valid_zip_codes = pl.read_json("data/sf_zipcodes.json")['zip_codes'].explode(empty_as_null=True)
bus = bus.filter(pl.col('postal_code').is_in(valid_zip_codes.implode()))

ins = pl.read_csv('data/ins.csv')
ins = ins.with_columns(
    pl.col('date').str.to_datetime(format='%m/%d/%Y %I:%M:%S %p').alias('timestamp'),
    pl.col('iid').str.split("_").list.get(0).cast(pl.Int64).alias('bid')
)

# This code is essential for the autograder to function properly. Do not edit.
ins_test = ins

<br/>

---

## 🔎 Question 1: Inspecting the Inspections

### 🚀 Question 1a

Let's start by looking again at the first 5 rows of `ins` to see what we're working with.

In [ ]:
ins.head(5)

To better understand how the scores have been allocated, let's examine how the maximum score varies for each type of inspection. 

Create a `DataFrame` object `ins_score_by_type` with two columns: `type`, containing each inspection type (e.g., New Construction, Routine - Unscheduled, etc.), and `max_score`, containing the highest score that type received. Additionally, order `ins_score_by_type` by `max_score` in descending order. 

**Hint:** `agg` will name its output column for you if you pass it as a keyword argument. See the `agg` [documentation](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.agg.html).

In [ ]:
ins_score_by_type = ...

ins_score_by_type

In [ ]:
grader.check("q1a")

<br/>

---

### 🚀 Question 1b

Given the variability of `ins['score']` observed in 1a, let's examine the inspection scores `ins['score']` further.

In [ ]:
ins['score'].value_counts(sort=True).head()

There are a large number of inspections with a score of -1. These are probably missing values. Let's see what types of inspections have scores and which do not (score of -1). 

- First, define a new column `Missing Score` in `ins` where each row maps to the string `"Yes"` if the `score` for that business is -1 and `"No"` otherwise. 

- Then, use `group_by` to find the number of inspections for every combination of `type` and `Missing Score`. Store these values in a new column `Count`.

- Finally, sort `ins_missing_score_group` by descending `Count`s. 
The result should be a `DataFrame` that looks like the one shown below.

**Hint**: `pl.when(...).then(...).otherwise(...)` ([documentation](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html)) builds a column conditionally, the same pattern you saw in HW 1.

*Hint 2*: `replace_strict` ([documentation](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.replace_strict.html)) also works if you would rather map the two Boolean values directly. 

<table border="1" class="dataframe">
  <caption>Beginning of the data frame you should obtain for question 1b.</caption>
  <thead>
    <tr>
      <th scope="col">type</th>
      <th scope="col">Missing Score</th>
      <th scope="col">Count</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Routine - Unscheduled</td>
      <td>No</td>
      <td>14031</td>
    </tr>
    <tr>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>

In [ ]:
ins = ...
ins_missing_score_group = ...


ins_missing_score_group

In [ ]:
grader.check("q1b")

<br/>

---

## 🚀 Question 2: Joining Data Across Tables

In this question, we will start to connect data across multiple tables. We will be using the `join` method.

<br/>

--- 

### 🚀 Question 2a

Let's figure out which restaurants had the lowest scores. Before we proceed, filter out missing scores from `ins` so that negative scores don't influence our results.

In [ ]:
ins = ins.filter(pl.col("score") > 0)

We'll start by creating a new `DataFrame` called `ins_named`. `ins_named` should be exactly the same as `ins`, except that it should have the name and address of every business, as determined by the `bus` `DataFrame`. 

**Hint**: Use the `DataFrame` method `join` to combine the `ins` `DataFrame` with the appropriate portion of the `bus` `DataFrame`. See the [documentation](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html) for guidance on how to use `join` to combine two `DataFrame` objects. The first few rows of the `ins_named` `DataFrame` are shown below:

<img src="images/2a.png" width="1080" alt="Dataframe with the columns 'iid,' 'date,' 'score,' 'type,' 'timestamp,' 'bid,' 'Missing Score,' 'name,' and 'address.'">

In [ ]:
ins_named = ...

ins_named.head()

In [ ]:
grader.check("q2a")

<br/>

--- 

### 🚀 Question 2b

Look at the 10 businesses in `ins_named` with the lowest scores. Order `ins_named` by each business's minimum score in ascending order. Use the business names in ascending order to break ties. The resulting `DataFrame` should look like the table below.

This one is pretty challenging! Don't forget to name the minimum-score column `min score`.

**Hint**: `agg` takes one expression per column you want back, and `alias` ([documentation](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.alias.html)) names each one. See the `agg` [documentation](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.agg.html). Additionally, when thinking about what aggregation functions to use, ask yourself: "*What value would be in the `name` column for each entry across the group? Can we select just one of these values to represent the whole group?*"

This question (2b) is a good candidate to try using an LLM to help you navigate the steps described above, but make sure you know exactly what's happening at each step!

As usual, **YOU SHOULD NOT USE LOOPS OR LIST COMPREHENSIONS**. Try to break down the problem piece by piece instead, gradually chaining together different `Polars` methods. Feel free to use more than one line!

<table border="1" class="dataframe">
  <caption>Beginning of the table you should obtain for question 2b</caption>
  <thead>
    <tr>
      <th scope="col">bid</th>
      <th scope="col">name</th>
      <th scope="col">min score</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>86718</td>
      <td>Lollipot</td>
      <td>45</td>
    </tr>
    <tr>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
  </tbody>
</table>

In [ ]:
ten_lowest_scoring = ... 

# DO NOT USE LIST COMPREHENSIONS OR LOOPS OF ANY KIND!!!


ten_lowest_scoring

In [ ]:
grader.check("q2b")

<br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 📊 Part 2: Visualization and Polars Potpourri

We will work with a visualization, build an analysis plan, and practice using LLMs to help us prepare data.

## 📝 Question 3: Making a complex plot

<img src="assets/asian-american-geography.png" width="1000" alt="Bubble scatter plot showing a slight negative relationship between states’ Asian-identifying share of high scorers and estimated attendance rates for white high scorers without legacy status.">

Consider the dataset used to produce the plot above. 

- The plot is Figure 3 of [this research paper](https://5harad.com/papers/college-admissions.pdf). 
- The caption will be helpful to read! 
- You might search through the paper to find definitions of words like "Ivy-11", "legacy", and "high-scorer".

**Note: We recommend that you do not use an LLM to help you with this problem.**

<!-- BEGIN QUESTION -->

### 🚀 Question 3a

What is the minimum number of rows needed to reproduce the plot above? Explain your choice.

If you need to make any assumptions, state them.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 🚀 Question 3b

What is the minimum number of columns needed to reproduce the plot? 

Additionally, provide a brief description of the data contained in each column.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 🚀 Question 3c

What is the granularity of the dataset used to produce the plot above?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 🚀 Question 3d

In your opinion, what is the granularity of the raw data that was "rolled-up" in order to produce the dataset plotted above? Try to get as fine-grained as possible. 

If you need to make any assumptions, state them.

*Hint: When is data about college applications first entered into an electronic database? You should use this first-stage data to answer the question.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 🚀 Question 3e

What are the essential columns in the raw data from part (d) that are required to produce the summarized data plotted above?

If you need to make any assumptions, state them.

*Hint: When application data is first entered into an electronic database, what information is recorded? What subset of information is referenced in the plot above?*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 🚀 Question 3f

Write an analysis plan for transforming the raw data from parts (d) and (e) into the dataset used to make the plot above. Your analysis plan should contain at least one "For each..." statement. 

If you need to make any assumptions, state them.

*Hint: You will find it helpful to keep track of multiple intermediate datasets and join them. Break this problem into smaller pieces. For example, how do you calculate the values on the x-axis? How do you calculate the values on the y-axis? What about the other dimensions of the plot?*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## 🏫 Question 4

In this question, you will partially reproduce the cleaned college admissions dataset we used during lecture. 

- Note that we don't provide precise steps for every operation you'll carry out in this problem. To become a data scientist, you need to learn to be scrappy and successful with only high-level instructions. 
- Of course, if you're struggling to proceed, come to office hours or post on Ed. We're here to help!

**Note: You will find it helpful to use an LLM to assist you on this problem as needed.** Remember, university students can get [free Gemini pro for a year](https://one.google.com/ai-student?g1_landing_page=75)!

### 🚀 Question 4a

 Head to the [UC Admissions by source school](https://www.universityofcalifornia.edu/about-us/information-center/admissions-source-school) page. Download the “Counts of fall freshmen by race/ethnicity” CSV data for UC Berkeley 2025 admits from California public high schools.

**Important! Once you download the CSV data, name the file `admitdata.csv` and upload to the `data` folder.**

- Throughout this question, you are going to have to look up how to do things like `How do I read a CSV file using Polars?` or `How do I count the number of rows in a dataframe?`. Scrappiness and self-help are key skills of a data scientist!

Assign your final dataframe to the variable `q4a`.

In [ ]:
q4a = ...


In [ ]:
grader.check("q4a")

### 🚀 Question 4b

Using an LLM to help you, or by searching the Polars documentation, write Polars code to remove all rows and columns from the Q4a dataset that contain data we did not use as part of our exploration in lecture. 

- Then, pivot the filtered data so it looks the same as the data on Slide 16 in Lecture 2. 

Finally, assign the resulting pivoted dataframe to the variable `q4b`.

- You can compare your data to a data preview from the slides to confirm that you have carried out this step correctly.

In [ ]:
# Keep only the columns used in lecture.
...
q4b = ...


In [ ]:
grader.check("q4b")

### 🚀 Question 4c

Next, head to the [California Department of Education website](https://www.cde.ca.gov/ds/) and download the data that contains the Grade 12 enrollment of each school in California. 

- Make sure to use the data from the most appropriate academic year given that our UC admissions data is from 2025.

- Read the data documentation to figure out which enrollment counts you should use for each school. You will have a lot of options to choose from.

- You will have to convert your data from `latin1` encoding to `utf-8` encoding.

- **For this subpart, please directly read the source URL of the data using `pl.read_csv`.**

Assign your final dataframe to the variable `q4c`.

In [ ]:
q4c = ...


In [ ]:
grader.check("q4c")

<!-- BEGIN QUESTION -->

### 🚀 Question 4d

In the next few parts, we will use the provided `data/hs-names-translator.csv` file to help us join the admissions data to the California HS enrollment data. 

- If you're interested in learning how this translator file was produced, come to instructor office hours!

If we're trying to reproduce the analysis from lecture, does it make sense to use a left join, right join, full join, or inner join to merge the UC admission dataset and the California HS enrollment dataset? Defend your choice in at least one sentence.

In [ ]:
translator = pl.read_csv("data/hs-name-translator.csv", null_values="NA")
translator.head(10)

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 🚀 Question 4e

In the lecture example, we had four schools with applications rates over 100%. One of them was "GIRLS ACADEMIC LEADERSHIP ACAD". Find this school in the translator file. 

Does it look like this school was matched correctly? If not, do you have a hypothesis as to why it could have been matched incorrectly? Write an explanation at least one sentence long.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 🚀 Question 4f

First, merge the UC admissions dataframe with the HS translator data using the appropriate type of join. 

- Then, merge the resulting dataframe with the California HS enrollment data using the appropriate type of join. 

- Finally, create a new column containing the application rate for each high school. 

To confirm that your merge was done correctly, you can reference any relevant data snippet shown in the lecture slides. 

- **The `grade_12` column used in the lecture dataset will have slightly different values than the correct answer to this question. As long as the values are close, you are all set!**

Assign your final dataframe to the variable `q4f`.

In [ ]:
...
q4f = ...



In [ ]:
grader.check("q4f")

<br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## 🎥 Part 3: SQL Fun: Movie Ratings

We will explore a miniature version of the [IMDb Dataset](https://www.imdb.com/interfaces/).

**Caution: Be careful with large SQL queries!!** You may need to reboot your Jupyter Hub instance if it stops responding. To avoid printing out 100k-sized tables, we've adjusted the display limit to ensure that the tables displayed are truncated to 20 rows (though they may contain more rows in reality).

In [ ]:
%config SqlMagic.displaylimit = 20
%config SqlMagic.style = 'DEFAULT'

Let's set up the SQL database.

<br/>

---

## ⚙️ Question 5 - Setup

Please run the cells below to set up your SQL database and the autograder.

In [ ]:
import duckdb

In [ ]:
# Run this cell to connect to the database
conn = duckdb.connect()
conn.query("INSTALL sqlite")
conn.query("ATTACH 'data/imdbmini.db' AS imdb (TYPE sqlite)")
conn.query("USE imdb")

In [ ]:
%sql duckdb:///data/imdbmini.db

Let's take a look at the table schemas:

In [ ]:
%%sql
-- just run this cell --
SELECT * FROM sqlite_master WHERE type='table';

From running the above cell, we see the database has 4 tables: `Name`, `Role`, `Rating`, and `Title`.

<details open>
    <summary>[<b>Click to Expand</b>] See descriptions of each table's schema below. You can also find it in the `schemas.txt` file, which is in the same directory as this notebook. We have only included descriptions for columns that could be of potential use in this homework. </summary>
    
**`Name`** – Contains the following information for names of people.
    
- nconst (BIGINT) - alphanumeric unique identifier of the name/person
- primaryName (VARCHAR) - name by which the person is most often credited
- birthYear (VARCHAR) - in YYYY format
- deathYear (VARCHAR) - in YYYY format
- primaryProfession (VARCHAR) – array of the top-3 professions of the person

    
    
**`Role`** – Contains the principal cast/crew for titles.
    
- tconst (BIGINT) - alphanumeric unique identifier of the title
- ordering (VARCHAR) - a number to uniquely identify rows for a given tconst
- nconst (BIGINT) - alphanumeric unique identifier of the name/person
- category (VARCHAR) - the category of job that person was in
- characters (VARCHAR) - the name of the character played if applicable, else '\\N'
    
**`Rating`** – Contains the IMDb rating and vote information for titles.
    
- tconst (BIGINT) - alphanumeric unique identifier of the title
- averageRating (VARCHAR) – weighted average of all the individual user ratings
- numVotes (VARCHAR) - number of votes (i.e., ratings) the title has received
    
**`Title`** - Contains the following information for titles.
    
- tconst (BIGINT) - alphanumeric unique identifier of the title
- titleType (VARCHAR) -  the type/format of the title
- primaryTitle (VARCHAR) -  the more popular title / the title used by the producers on promotional materials at the point of release
- isAdult (VARCHAR) - 0: non-adult title; 1: adult title
- startYear (VARCHAR) - represents the release year of a title.
- runtimeMinutes (VARCHAR) - primary runtime of the title, in minutes
- genres (VARCHAR) – array that includes up to three genres associated with the title
    
</details>

<br/><br/>
From the above descriptions, we can conclude the following:
* `Name.nconst` and `Title.tconst` are primary keys of the `Name` and `Title` tables, respectively.
* `Role.nconst` and `Role.tconst` are **foreign keys** that point to `Name.nconst` and `Title.tconst`, respectively.

Keep in mind that you can directly write your query in place of the ellipsis under `%%sql --save query_q`. **Please do not edit this line.** 

For example, we can set `query_example` to the output of the following SQL query and directly see what it contains:

In [ ]:
%%sql --save query_example
-- This is a one-line SQL comment.
/* This is a multi-line
   SQL comment. */
SELECT * 
FROM name
LIMIT 15;

<br/>

---

### 🚀 Question 5a  
Let's determine whether our database includes information going back to the early days of cinema, or just more recent data.

List the **5 oldest movie titles** by `startYear` and then `primaryTitle` both in **ascending** order. The output should contain the `startYear`, `primaryTitle`, and `titleType`. In this homework, we define a movie as having `titleType='movie'`. Keep this in mind for later questions as well.

In [ ]:
%%sql --save query_q5a

...

In [ ]:
# Run this cell for grading purposes. 
# No further action is required. 
query = %sqlcmd snippets query_q5a
res_q5a = conn.query(query).pl()

In [ ]:
grader.check("q5a")

<br/>

---

### 🚀 Question 5b

Next, let's calculate the distribution of movies by year. Write a query that returns the **total** number of movie titles for each `startYear` in the `Title` table as `total`. Order your final results by the `startYear` in **ascending** order. As in `q5a`, remember that movies are defined as having `titleType='movie'`.

The first few records of the table should look like the following (but you should compute the entire table):


|startYear|total|
|------:|-----:|
| 1915|1|
| 1920|1|
| 1921|1|
| 1922|1|
| ...|...|

In [ ]:
%%sql --save query_q5b

...

In [ ]:
# Run this cell for grading purposes. 
# No further action is required. 
query = %sqlcmd snippets query_q5b
res_q5b = conn.query(query).pl()

In [ ]:
grader.check("q5b")

<br/>

The following cell should generate an interesting plot of the number of movies that premiered each year. Notice there are fewer movies premiering from the 1920s to the late 1940s. Why might that be? *This question is rhetorical; you do not need to write your answer anywhere.*

In [ ]:
# Run this call to generate the bar plot; no further action is needed
px.bar(res_q5b, x="startYear", y="total", 
        title="Number of movies premiered each year", 
        width=900, height=400)

<br/>

---

## 🎬 Question 6

Write a SQL query to determine the **movie actors** with the highest total number of movies. The term **movie actor** is defined as anyone with an "actor" or "actress" job category in a "movie" title type released after 1980. Your SQL query should output exactly two fields named `name` (the movie actor’s name) and `total` (the number of movies the movie actor appears in). Order the records by `total` in descending order, and break ties by ordering by `name` in ascending order. **Only include the first 20 rows in your final query.**

Your result should look something like this (but without `????`):

| name | total |
|-----:|-----:|
| ???? | 58 |
| ???? | 54 |
| ???? | 53 |
| ???? | 49 |
| ???? | 46 |
| ???? | 43 |
| ???? | 41 |
| ???? | 40 |
| ???? | 40 |
| ???? | 39 |

**Notes**: 
* **The query should take < 2 minutes to run.**
* Sometimes Python gets confused and colors some SQL queries red; *don't worry if the SQL coloring doesn't match what you'd expect*. As long as it runs, it's fine.

**Hints**:

* Before writing your query, you may wish to review the table descriptions given at the start of the assignment to determine where the information you need is stored
* If you want to include a non-aggregate field in the `SELECT` clause, it must also be included in the `GROUP BY` clause.
* When using multiple conditions in a `WHERE` clause, pay attention to the order of operations.

In [ ]:
%%sql --save query_q6

...

In [ ]:
# Run this cell for grading purposes. 
# No further action is required. 
query = %sqlcmd snippets query_q6
res_q6 = conn.query(query).pl()

In [ ]:
grader.check("q6")

<br/>

---

## 🔁 Question 7: The `CASE` Keyword

The `rating` table has the `numVotes` and the `averageRating` for each title. A movie is considered a **"big hit**" if there are more than 100,000 votes for the movie. Which `movie` titles were **"big hits"**? Construct a query that generates the following result:

| isBigHit | total |
|-----:|-----|
| no | ???? |
| yes | ???? |

Where `????` is replaced with the correct values. The row with `no` should have the count for how many movies **are not** big hits, and the row with `yes` should have the count of how many movies **are** big hits.

**Hints**:

* Check the data types of `numVotes` before performing any arithmetic operations.
* You will need to use some type of `JOIN`.
* You may also consider using a `CASE` statement:
    ```
    CASE 
        WHEN ... THEN ...
        ELSE ... 
    END
    ```
    </br>
    
  `CASE` statements are the SQL equivalent of `Python` `if... elif... else` statements. To read up on `CASE`, take a look at the following links:
    - https://mode.com/sql-tutorial/sql-case/
    - https://www.w3schools.com/sql/sql_ref_case.asp

In [ ]:
%%sql --save query_q7

...

In [ ]:
# Run this cell for grading purposes. 
# No further action is required. 
query = %sqlcmd snippets query_q7
res_q7 = conn.query(query).pl()

In [ ]:
grader.check("q7")

<br/>

---

## 🚀 Question 8 (optional)

Write a SQL query to determine the movie producers with the highest average ratings across all of their movies. Define a **"movie producer"** as anyone with a `producer` job category role in a `movie` title type. Construct a query that generates a table consisting of the **producer's name** (as `name`) and their **average producer rating** (as `producerRating`), computed by rescaling the ratings for movies they produced by the number of votes received by each movie. After rescaling the score for each movie, divide the summation of all rescaled ratings by the total number of votes received by each producer. The formula is below:

$$
\text{producerRating} = 
\frac{\sum_m (\texttt{averageRating}[m] * \texttt{numVotes}[m] )}{\sum_m \texttt{numVotes}[m]}
$$

To make clear the summation, "m" refers to a particular movie that a producer worked on. Thus, the summation over "m" refers to the summation across all movies that a particular producer contributed to.

In addition to the above, only consider ratings where there are **at least 22,500** votes and only consider movie producers that have **at least 20 rated movies**. Present the producers with the **top 10** `producerRating` in **descending** order and break ties alphabetically using the producer's name.

The results should look something like this but without the `????`, and with higher rating precision.

| name            | producerRating |
|-----------------|----------------|
| ???             | 7.76...        |
| ???             | 7.62...        |
| ???             | 7.59...        |
| ???             | 7.43...        |
| ???             | 7.41...        |
| ???             | 7.35...        |
| ???             | 7.30...        |
| ???             | 7.27...        |
| ???             | 7.25...        |
| ???             | 7.24...        |

**Hint**: Check the data types of `numVotes` and `averageRating` before performing any arithmetic operations.

**Notes**:
* This question is **optional!** You will not lose credit for failing the public tests in this question.
* ***The query should take < 3 minutes to run.***
* DO NOT cast `averageRating` **as an integer**. Doing so reduces the precision of the resulting values, so your table may not match up exactly with what is shown below.
* If a producer has multiple `role` listings for a movie, then that movie will have a bigger impact on the overall average (this is desired).

In [ ]:
%%sql --save query_q8

...

In [ ]:
# Run this cell for grading purposes. 
# No further action is required. 
query = %sqlcmd snippets query_q8
res_q8 = conn.query(query).pl()

In [ ]:
grader.check("q8")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Congratulations! You have finished Homework 2! ##

Scout says hi :)

<img src = "images/Scout.PNG" width = "400" class="center" alt="Happy dog">

## Course Content Feedback

If you have any feedback about this assignment or about any of our other weekly, weekly assignments, lectures, or discussions, please fill out the [Course Content Feedback Form](https://docs.google.com/forms/d/e/1FAIpQLSfN9C-RJoe9hD8G2sbd1reJQh5H4WwKLFFrEt4DeQUBKmToJQ/viewform?usp=dialog). Your input is valuable in helping us improve the quality and relevance of our content to better meet your needs and expectations!

## Submission Instructions

Below, you will see a cell. Running this cell will automatically generate a zip file with your autograded answers. Once you submit this file to the HW 2 Coding assignment on Pensive, Pensive will automatically submit a PDF file with your written answers to the HW 2 Coding Written assignment. If you run into any issues when running this cell, feel free to check this [section](https://ds100.org/debugging-guide/autograder-pensive/) in the Data 100 Debugging Guide.

**Important**: Please check that your written responses were generated and submitted correctly to the HW 2 Coding Written Assignment.

**You are responsible for ensuring your submission follows our requirements and that the PDF for HW 2 written answers was generated/submitted correctly. We will not be granting regrade requests nor extensions to submissions that don't follow instructions.** If you encounter any difficulties with submission, please don't hesitate to reach out to staff prior to the deadline.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)